In [1]:
pip install zstandard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import io
import json
import zstandard as zstd

In [6]:
folder_path = "subreddits24"
target_file = os.path.join(folder_path, files[0])  # ia primul fișier din listă
print(f"Inspectăm fișierul: {target_file}\n")

with open(target_file, "rb") as fh:
  dctx = zstd.ZstdDecompressor(max_window_size=2147483648)
  with dctx.stream_reader(fh) as reader:
    text_stream = io.TextIOWrapper(reader, encoding="utf-8")

    count = 0
    while count < 10:
      line = text_stream.readline()
      if not line:
        break

      data = json.loads(line)

      # Verificăm dacă e comentariu ('body') sau postare ('title' / 'selftext')
      text_content = (
          data.get("body") or data.get("selftext") or data.get("title")
      )

      # Ignorăm comentariile șterse când inspectăm
      if text_content in ["[deleted]", "[removed]"]:
        continue

      count += 1
      print(f"{'='*20} Exemplul {count} {'='*20}")
      print(f"Autor: {data.get('author')}")
      print(f"Subreddit: {data.get('subreddit')}")
      print(f"Score: {data.get('score')}")
      print(f"Text:\n{text_content}")
      print("\nToate cheile disponibile în JSON:", list(data.keys()))
      print("\n")

Inspectăm fișierul: subreddits24\freeworldnews_comments.zst

==================== Exemplul 1 ====================
Autor: andsendunits
Subreddit: freeworldnews
Score: 2
Text:
That is excellent to read. 

Toate cheile disponibile în JSON: ['subreddit_id', 'author_flair_css_class', 'link_id', 'controversiality', 'id', 'parent_id', 'score', 'body', 'ups', 'distinguished', 'edited', 'retrieved_on', 'author', 'subreddit', 'created_utc', 'gilded', 'author_flair_text', 'stickied']


==================== Exemplul 2 ====================
Autor: autotldr
Subreddit: freeworldnews
Score: 2
Text:
This is the best tl;dr I could make, [original](http://www.latimes.com/world/asia/la-fg-japan-ghost-ships-20160410-story.html) reduced by 89%. (I'm a bot)
*****
&gt; Shizuo Kakutani sees no great mystery in the things that wash ashore in Monzen, his quiet fishing village on the Sea of Japan - the fishing boats ravaged by fierce winter storms, the Chinese garbage carried to land by the strong winds, the occas

In [9]:
# ============================================================
#  O POSTARE + TOATE COMENTARIILE EI (afișate ca arbore)
# ============================================================
import html
from collections import defaultdict
from datetime import datetime, timezone

SUBREDDIT       = "worldnewsvideo"   # schimbă cu orice subreddit din subreddits24/
POST_ID         = None               # pune un id anume (ex: "jzsgkm") sau None = alege automat
MIN_COMENTARII  = 5                  # la alegerea automată: postări cu între MIN și MAX comentarii
MAX_COMENTARII  = 15
ARATA_TOT_JSON  = True              # True = afișează TOATE câmpurile fiecărui obiect


def citeste_zst(cale):
    """Citește un .zst linie cu linie, fără să-l dezarhiveze pe disc.
    Fiecare linie = un obiect JSON (o postare sau un comentariu)."""
    with open(cale, "rb") as fh:
        dctx = zstd.ZstdDecompressor(max_window_size=2**31)
        with dctx.stream_reader(fh) as reader:
            for line in io.TextIOWrapper(reader, encoding="utf-8"):
                yield json.loads(line)


def curata(text):
    """&gt; → >, &amp; → & (de 2 ori, pt. cazuri ca &amp;#039;)."""
    return html.unescape(html.unescape(text or ""))


def data_ora(ts):
    return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%d %H:%M")


fis_posts    = os.path.join("subreddits24", f"{SUBREDDIT}_submissions.zst")
fis_comments = os.path.join("subreddits24", f"{SUBREDDIT}_comments.zst")

# ---------- PASUL 1: citim postările (submissions) ----------
candidati = {}                                  # id postare → dicționarul postării
for s in citeste_zst(fis_posts):
    if POST_ID is not None:
        if s["id"] == POST_ID:
            candidati[s["id"]] = s
            break
    elif MIN_COMENTARII <= s.get("num_comments", 0) <= MAX_COMENTARII:
        candidati[s["id"]] = s
        if len(candidati) >= 200:               # 200 de candidați ajung
            break
print(f"Postări candidate: {len(candidati)}")

# ---------- PASUL 2: citim comentariile și le grupăm pe postare ----------
# link_id = "t3_<id postare>"  → de aici știm la ce postare e comentariul
comentarii_pe_postare = defaultdict(list)
for c in citeste_zst(fis_comments):
    id_postare = c["link_id"].replace("t3_", "")
    if id_postare in candidati:
        comentarii_pe_postare[id_postare].append(c)

# alegem postarea cu cele mai multe comentarii „reale” (nu [deleted]/[removed])
def nr_reale(pid):
    return sum(c["body"] not in ("[deleted]", "[removed]") for c in comentarii_pe_postare[pid])

pid        = max(comentarii_pe_postare, key=nr_reale)
postare    = candidati[pid]
comentarii = comentarii_pe_postare[pid]

# ---------- PASUL 3: afișăm POSTAREA ----------
print("\n" + "=" * 70)
print("📌 POSTARE PRINCIPALĂ (din _submissions.zst)")
print("=" * 70)
print(f"id            : {postare['id']}        (în comentarii apare ca link_id = t3_{postare['id']})")
print(f"titlu         : {curata(postare.get('title'))}")
print(f"selftext      : {curata(postare.get('selftext')) or '(gol - e link/video)'}")
print(f"url           : {postare.get('url')}")
print(f"autor         : {postare.get('author')}")
print(f"score         : {postare.get('score')}   upvote_ratio: {postare.get('upvote_ratio')}")
print(f"num_comments  : {postare.get('num_comments')}   (găsite efectiv în arhivă: {len(comentarii)})")
print(f"data          : {data_ora(postare['created_utc'])}")
print(f"link reddit   : https://reddit.com{postare.get('permalink', '')}")
if ARATA_TOT_JSON:
    print("\n--- TOATE câmpurile postării ---")
    print(json.dumps(postare, indent=2, ensure_ascii=False))

# ---------- PASUL 4: afișăm COMENTARIILE ca arbore ----------
# parent_id = "t3_..." → răspuns direct la postare
# parent_id = "t1_..." → răspuns la ALT comentariu
copii = defaultdict(list)
for c in comentarii:
    copii[c["parent_id"]].append(c)
for lista in copii.values():
    lista.sort(key=lambda c: int(c["created_utc"]))


def afiseaza(parinte, nivel):
    for c in copii.get(parinte, []):
        ind  = "    " * nivel
        text = curata(c["body"]).replace("\n", " ")
        nr_cuv = len(text.split())
        print(f"{ind}├── [{c['id']}] {c['author']}  | score={c['score']}  "
              f"| controv={c.get('controversiality')}  | {nr_cuv} cuvinte  | {data_ora(c['created_utc'])}")
        print(f"{ind}│     parent_id = {c['parent_id']}"
              f"   ({'răspunde POSTĂRII' if c['parent_id'].startswith('t3_') else 'răspunde comentariului ' + c['parent_id'][3:]})")
        print(f"{ind}│     \"{text}\"")
        if ARATA_TOT_JSON:
            print(json.dumps(c, indent=2, ensure_ascii=False))
        afiseaza("t1_" + c["id"], nivel + 1)   # copiii acestui comentariu


print("\n" + "=" * 70)
print(f"💬 COMENTARII (din _comments.zst) — {len(comentarii)} în total")
print("=" * 70)
afiseaza("t3_" + pid, 0)

# comentarii al căror părinte lipsește din arhivă (rar, dar se întâmplă)
ids = {"t1_" + c["id"] for c in comentarii} | {"t3_" + pid}
orfani = [c for c in comentarii if c["parent_id"] not in ids]
if orfani:
    print(f"\n⚠️ {len(orfani)} comentarii au părintele lipsă din arhivă: {[c['id'] for c in orfani]}")

Postări candidate: 200

📌 POSTARE PRINCIPALĂ (din _submissions.zst)
id            : jykmp5        (în comentarii apare ca link_id = t3_jykmp5)
titlu         : Anti-choice supporter is given an interesting hypothetical
selftext      : (gol - e link/video)
url           : https://v.redd.it/hhv3giozco061
autor         : PlenitudeOpulence
score         : 398   upvote_ratio: 0.98
num_comments  : 15   (găsite efectiv în arhivă: 17)
data          : 2020-11-21 23:07
link reddit   : https://reddit.com/r/worldnewsvideo/comments/jykmp5/antichoice_supporter_is_given_an_interesting/

--- TOATE câmpurile postării ---
{
  "all_awardings": [],
  "allow_live_comments": false,
  "archived": true,
  "author": "PlenitudeOpulence",
  "author_created_utc": 1531238952,
  "author_flair_background_color": "",
  "author_flair_css_class": null,
  "author_flair_richtext": [],
  "author_flair_template_id": null,
  "author_flair_text": "Primary 🏛",
  "author_flair_text_color": "dark",
  "author_flair_type": "text",

In [ ]:
T